# 零样本跨环境评估分析

分析在未见 CALVIN 环境 D 上的零样本泛化结果，
对比基线 (单环境 A) 与联合训练 (多环境 A+B+C) 的 ACT 策略。

In [ ]:
import sys
sys.path.insert(0, '..')

import json
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

%matplotlib inline
plt.rcParams.update({
    'figure.dpi': 150,
    'font.size': 11,
    'axes.titlesize': 13,
})

In [ ]:
# 加载评估结果
EVAL_PATH = Path("../outputs/logs/eval_results.json")

if EVAL_PATH.exists():
    with open(EVAL_PATH, encoding="utf-8") as f:
        results = json.load(f)
    print(f"加载 {results['env']} 环境评估结果, {results['num_episodes']} episodes")
else:
    print("未找到评估结果，使用占位数据。")
    results = {
        "env": "D",
        "num_episodes": 100,
        "baseline": {"success_rate": 0.45, "avg_steps": 180, "num_successes": 45},
        "joint": {"success_rate": 0.72, "avg_steps": 220, "num_successes": 72},
    }

In [ ]:
baseline = results["baseline"]
joint = results["joint"]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle(f'环境 {results["env"]} 零样本泛化评估',
             fontsize=14, fontweight='bold')
colors = {'baseline': '#2196F3', 'joint': '#FF5722'}
models = ['Baseline\n(Env A)', 'Joint\n(Env A+B+C)']

# 成功率
ax = axes[0]
sr = [baseline['success_rate'] * 100, joint['success_rate'] * 100]
bars = ax.bar(models, sr, color=[colors['baseline'], colors['joint']])
ax.set_ylabel('成功率 (%)')
ax.set_title('任务成功率')
for bar, val in zip(bars, sr):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f'{val:.1f}%', ha='center', fontweight='bold', fontsize=13)
ax.set_ylim(0, max(sr) * 1.4 + 5)
ax.grid(True, alpha=0.3, axis='y')

# 平均步数
ax = axes[1]
steps = [baseline.get('avg_steps', 0), joint.get('avg_steps', 0)]
bars = ax.bar(models, steps, color=[colors['baseline'], colors['joint']])
ax.set_ylabel('平均步数')
ax.set_title('Episode 长度')
for bar, val in zip(bars, steps):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f'{val:.0f}', ha='center', fontweight='bold', fontsize=13)
if max(steps) > 0:
    ax.set_ylim(0, max(steps) * 1.3)
ax.grid(True, alpha=0.3, axis='y')

# 成功次数
ax = axes[2]
ns = [baseline.get('num_successes', 0), joint.get('num_successes', 0)]
total = results.get('num_episodes', 100)
bars = ax.bar(models, ns, color=[colors['baseline'], colors['joint']])
ax.set_ylabel(f'成功次数 / {total}')
ax.set_title('成功次数')
for bar, val in zip(bars, ns):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{val}', ha='center', fontweight='bold', fontsize=13)
ax.set_ylim(0, total * 1.2)
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('../outputs/figures/zero_shot_results.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 结果分析
sr_delta = (joint['success_rate'] - baseline['success_rate']) * 100
steps_delta = joint.get('avg_steps', 0) - baseline.get('avg_steps', 0)

print("=" * 60)
print("零样本泛化分析")
print("=" * 60)
print(f"测试环境: {results['env']} (训练中未见)")
print()
print(f"成功率差异:    {sr_delta:+.1f}%")
print(f"平均步数差异:  {steps_delta:+.1f}")
print()

if sr_delta > 5:
    print("结论: 联合训练 (A+B+C) 在未见环境上显著优于单环境基线。")
    print("这表明多环境数据帮助 ACT 策略学习了更鲁棒的视觉表征，")
    print("动作分块机制的时序一致性先验在应对视觉分布偏移时起到了隐式正则化作用。")
elif sr_delta > -5:
    print("结论: 二者性能相当。ACT 策略从多环境数据中获益有限，")
    print("表明单环境已提供足够的视觉多样性。")
else:
    print("结论: 基线优于联合训练，可能存在环境间负迁移，")
    print("或模型容量不足以捕捉多环境分布。")